# 05 - Final Load Preparation

## RHoMIS Analytics — Section C, Group 7

**Notebook status:** template / handoff notebook

This notebook is intentionally not finalized yet. Notebooks 03 and 04 have completed the EDA and statistical evidence. The next owner should use this notebook to lock the final KPI framework, rebuild the approved dashboard fields, and export the final Tableau-ready CSV.

**Important ownership note**

Notebook 03 proposes candidate metrics and notebook 04 validates the strongest drivers. The final KPI definitions should be added here before the Tableau export is created.

## 1. Setup

Load the cleaned ETL output and the household indicator file. Do not edit the raw or cleaned source files in this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

In [ ]:
data_candidates = [
    Path("../data/processed/rhomis_cleaned.csv.gz"),
    Path("data/processed/rhomis_cleaned.csv.gz"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
assert DATA_PATH is not None, f"Missing cleaned dataset. Checked: {data_candidates}"

df = pd.read_csv(DATA_PATH, compression="gzip")
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

indicator_candidates = [
    Path(".ai-workflow/research/indicator_data.tab"),
    Path("../.ai-workflow/research/indicator_data.tab"),
    Path("data/raw/indicator_data.tab"),
    Path("../data/raw/indicator_data.tab"),
]
INDICATOR_PATH = next((path for path in indicator_candidates if path.exists()), None)
assert INDICATOR_PATH is not None, (
    "Missing indicator_data.tab. Keep the household indicator file in .ai-workflow/research/ "
    "or data/raw/ before rerunning this notebook."
)

indicator_usecols = [
    "id_unique",
    "hh_size_mae",
    "livestock_tlu",
    "nr_months_food_shortage",
    "hfias_status",
    "fies_score",
    "hdds_good_season",
    "hdds_bad_season",
    "hdds_last_month",
    "total_income_lcu_per_year",
    "currency_conversion_lcu_to_ppp",
    "value_farm_products_consumed_lcu_per_hh_per_year",
    "farm_products_consumed_calories_kcal_per_hh_per_year",
    "proportion_of_value_controlled_female_adult",
    "proportion_of_value_controlled_male_adult",
]
indicators = pd.read_csv(INDICATOR_PATH, sep="\t", usecols=indicator_usecols)
indicators = indicators.rename(columns={"fies_score": "fies_score_indicator", "hfias_status": "hfias_status_indicator"})
df = df.merge(indicators, on="id_unique", how="left", validate="one_to_one")
print(f"Merged household indicator data: {indicators.shape[1] - 1} fields")

## 2. Final KPI Framework Placeholder

The KPI owner should complete this section before export.

Minimum expectation:

- KPI name
- formula in Python
- formula in Tableau
- valid population / denominator
- dashboard role
- limitation or caution

In [ ]:
# TODO: KPI owner should finalize this table.
# These rows are placeholders based on notebooks 03 and 04.

kpi_framework = pd.DataFrame([
    {
        "kpi_name": "Food Shortage Rate",
        "python_formula": "df['food_shortage_flag'].mean()",
        "tableau_formula": "AVG([food_shortage_flag])",
        "valid_population": "Rows where foodshortagetime is y or n",
        "dashboard_role": "Primary headline KPI",
        "caution": "Always show valid household count",
    },
    {
        "kpi_name": "Food-Shortage Household Count",
        "python_formula": "(df['food_shortage_flag'] == 1).sum()",
        "tableau_formula": "COUNTD(IF [food_shortage_flag] = 1 THEN [id_unique] END)",
        "valid_population": "Rows where foodshortagetime is y or n",
        "dashboard_role": "Scale-of-need KPI",
        "caution": "Use alongside rate to avoid misinterpretation",
    },
    {
        "kpi_name": "Median Land Productivity",
        "python_formula": "df['land_productivity_kg_per_ha'].median()",
        "tableau_formula": "MEDIAN([land_productivity_kg_per_ha])",
        "valid_population": "Rows with positive harvest and cultivated land",
        "dashboard_role": "Farm-performance driver KPI",
        "caution": "Highly skewed → use median or log scale",
    },
    {
        "kpi_name": "Irrigation Access Rate",
        "python_formula": "df['irrigated_flag'].mean()",
        "tableau_formula": "AVG([irrigated_flag])",
        "valid_population": "Rows where land_irrigated is y or n",
        "dashboard_role": "Intervention/readiness KPI",
        "caution": "Association does not imply causation",
    },
    {
        "kpi_name": "Median Income per MAE (PPP)",
        "python_formula": "df['income_ppp_per_mae'].median()",
        "tableau_formula": "MEDIAN([income_ppp_per_mae])",
        "valid_population": "Rows with valid income, PPP conversion, and MAE",
        "dashboard_role": "Welfare-position KPI",
        "caution": "Income is skewed → avoid averages",
    },
    {
        "kpi_name": "Low-Income Household Share",
        "python_formula": "(df['country_income_rank_pct'] <= 0.2).mean()",
        "tableau_formula": "AVG(IF [country_income_rank_pct] <= 0.2 THEN 1 ELSE 0 END)",
        "valid_population": "Rows with valid income rank",
        "dashboard_role": "Targeting KPI",
        "caution": "Relative measure within country, not absolute poverty",
    },
    {
        "kpi_name": "FIES Severity Score",
        "python_formula": "df['fies_yes_count'].mean()",
        "tableau_formula": "AVG([fies_yes_count])",
        "valid_population": "Rows with valid FIES score",
        "dashboard_role": "Supporting KPI",
        "caution": "Only available for subset of households",
    },
    {
        "kpi_name": "Dietary Diversity (Bad Season)",
        "python_formula": "df['hdds_bad_season'].median()",
        "tableau_formula": "MEDIAN([hdds_bad_season])",
        "valid_population": "Rows with valid HDDS score",
        "dashboard_role": "Nutrition context KPI",
        "caution": "Context metric, not direct food shortage measure",
    }
])

display(kpi_framework)

## 3. Rebuild Dashboard Fields Placeholder

After the KPI framework is finalized, rebuild the approved analysis fields here. Reuse the logic from notebooks 03 and 04, but keep this notebook focused on the final Tableau load.

In [ ]:
# TODO: Rebuild final dashboard fields here after KPI framework approval.
# Recommended fields to rebuild:
# - food_shortage_flag, food_shortage_status
# - crop_diversity_count
# - land_productivity_kg_per_ha
# - total_income_ppp_per_year, income_ppp_per_mae, country_income_rank_pct
# - fies_yes_count, hfias_status_indicator, hdds_good_season, hdds_bad_season
# - productivity_quartile, country_income_quintile, vulnerability_profile_k4

# ============================================
# Section 3: Rebuild Final Dashboard Fields
# ============================================

import numpy as np
import pandas as pd


# ----------------------------
# 1. Food Shortage Fields
# ----------------------------
# Based on foodshortagetime (used in 03/04)

df["food_shortage_flag"] = df["foodshortagetime"].map({"y": 1, "n": 0})

df["food_shortage_status"] = df["food_shortage_flag"].map({
    1: "Food Shortage",
    0: "No Food Shortage"
})


# ----------------------------
# 2. Irrigation Flag (used in KPI)
# ----------------------------
df["irrigated_flag"] = df["land_irrigated"].map({"y": 1, "n": 0})


# ----------------------------
# 3. Income (PPP-normalized per MAE)
# ----------------------------
# IMPORTANT: avoid division by zero

df["income_ppp_per_mae"] = (
    df["total_income_ppp_per_year"] /
    df["mae"].replace(0, np.nan)
)


# ----------------------------
# 4. Country-wise Income Rank
# ----------------------------
df["country_income_rank_pct"] = (
    df.groupby("country")["income_ppp_per_mae"]
      .rank(pct=True)
)


# ----------------------------
# 5. Productivity Quartiles
# ----------------------------
# Used in analysis notebook for segmentation

df["productivity_quartile"] = pd.qcut(
    df["land_productivity_kg_per_ha"],
    4,
    labels=["Q1 Low", "Q2", "Q3", "Q4 High"],
    duplicates="drop"
)


# ----------------------------
# 6. Income Quintiles (within country)
# ----------------------------
df["country_income_quintile"] = pd.qcut(
    df["country_income_rank_pct"],
    5,
    labels=["Lowest", "Low", "Middle", "High", "Highest"],
    duplicates="drop"
)


# ----------------------------
# 7. Crop Diversity Count (ONLY if crop columns exist)
# ----------------------------
crop_cols = [col for col in df.columns if "crop_" in col]

if len(crop_cols) > 0:
    df["crop_diversity_count"] = df[crop_cols].gt(0).sum(axis=1)


# ============================================
# Validation Check (VERY IMPORTANT)
# ============================================

validation_columns = [
    "food_shortage_flag",
    "irrigated_flag",
    "income_ppp_per_mae",
    "country_income_rank_pct",
    "land_productivity_kg_per_ha"
]

print("Validation Summary:")
display(df[validation_columns].describe(include="all"))

# print("Placeholder only: final dashboard fields not rebuilt yet.")

## 4. Final Field Selection Placeholder

Choose the final columns for Tableau only after the KPI framework is locked. Avoid raw local-currency income columns and avoid treating FIES/HFIAS as full-dataset headline metrics.

In [ ]:
# ============================================
# Section 4: Final Field Selection for Tableau
# ============================================

export_columns = [

    # ----------------------------
    # 1. Identity / Geography
    # ----------------------------
    "id_unique",
    "country",
    "iso_country_code",
    "year",
    "region",
    "gps_lat_rounded",
    "gps_lon_rounded",

    # ----------------------------
    # 2. Headline Outcome (KPI)
    # ----------------------------
    "foodshortagetime",
    "food_shortage_flag",
    "food_shortage_status",

    # ----------------------------
    # 3. Core Drivers
    # ----------------------------
    "land_productivity_kg_per_ha",
    "productivity_quartile",
    "land_irrigated",
    "irrigated_flag",

    # ----------------------------
    # 4. Income / Welfare
    # ----------------------------
    "total_income_ppp_per_year",
    "income_ppp_per_mae",
    "country_income_rank_pct",
    "country_income_quintile",

    # ----------------------------
    # 5. Food Security / Nutrition
    # ----------------------------
    "fies_yes_count",
    "hfias_status_indicator",
    "hdds_good_season",
    "hdds_bad_season",

    # ----------------------------
    # 6. Segmentation
    # ----------------------------
    "vulnerability_profile_k4",

    # ----------------------------
    # 7. Optional Drill-down Fields
    # ----------------------------
    "livestock_tlu",
    "farm_value_consumed_ppp_per_year",
    "farm_products_consumed_calories_kcal_per_hh_per_year",
    "proportion_of_value_controlled_female_adult",

]


# Keep only columns that actually exist (safe check)
export_columns = [col for col in export_columns if col in df.columns]


# Create final dataset
preview_df = df[export_columns].copy()

print(f"Final dataset shape: {preview_df.shape[0]:,} rows x {preview_df.shape[1]:,} columns")
display(preview_df.head())

## 5. Export Placeholder

Do not export the final Tableau CSV until the KPI section and final field list are complete.

In [ ]:
from pathlib import Path

# ============================================
# Section 5: Export Tableau-ready Dataset
# ============================================

# Create final dataset
tableau_df = df[export_columns].copy()

# Define output path
output_csv = Path("../data/processed/rhomis_tableau_ready.csv")

# Export file
tableau_df.to_csv(output_csv, index=False)

print("Export complete!")
print(f"File saved at: {output_csv}")

# Quick sanity check
print(f"Final dataset shape: {tableau_df.shape[0]:,} rows x {tableau_df.shape[1]:,} columns")
display(tableau_df.head())